# Notebook 06: Structured Handoffs (CCA Pattern 6)

**CCA Pattern:** Structured EscalationRecord via `tool_choice` enforcement vs. raw conversation dump

This notebook demonstrates the final CCA pattern: when an agent escalates to a human, the handoff payload must be a **structured EscalationRecord** — never a raw conversation dump filled with API-internal `tool_use` blocks.

Scenario: C003 Carol Martinez, \$600 refund request (triggers mandatory escalation: amount > $500).

## Setup

In [ ]:
import json
import sys
from pathlib import Path

# Add project root so notebooks.helpers and customer_service are importable
sys.path.insert(0, str(Path(".").resolve()))

import anthropic
from helpers import compare_results, print_usage

from customer_service.agent import (
    build_callbacks,
    get_system_prompt,
    run_agent_loop,
)
from customer_service.anti_patterns import format_raw_handoff
from customer_service.services.audit_log import AuditLog
from customer_service.services.container import ServiceContainer
from customer_service.services.customer_db import CustomerDatabase
from customer_service.services.escalation_queue import EscalationQueue
from customer_service.services.financial_system import FinancialSystem
from customer_service.services.policy_engine import PolicyEngine

In [ ]:
def make_services() -> ServiceContainer:
    """Create a fresh ServiceContainer with seed customer data."""
    from customer_service.data.customers import CUSTOMERS

    return ServiceContainer(
        customer_db=CustomerDatabase(CUSTOMERS),
        policy_engine=PolicyEngine(),
        financial_system=FinancialSystem(),
        escalation_queue=EscalationQueue(),
        audit_log=AuditLog(),
    )


client = anthropic.Anthropic()

In [ ]:
# C003: Carol Martinez, $600 refund — canonical scenario for escalation
# amount > $500 triggers mandatory escalation regardless of tier
user_message = (
    "Hi, I need to return my purchase. My customer ID is C003 and my order is O003. "
    "The item was defective and I want a refund for the $600 I paid."
)

print("Customer: C003 (Carol Martinez, Regular tier)")
print("Order: O003 — $600 refund request")
print("Expected: escalation (amount > $500 threshold)")
print(f"\nMessage: {user_message}")

## Anti-Pattern: Raw Conversation Dump

<div style="border-left: 4px solid #dc3545; padding: 12px 16px; background: #fff5f5; margin: 8px 0;">
<strong>What's wrong:</strong> Dumping the full <code>messages</code> list as JSON gives the human agent thousands of characters of API-internal noise — <code>tool_use</code> blocks with tool IDs, <code>tool_result</code> blocks with JSON payloads, assistant reasoning fragments. None of this is useful to a human agent. The signal-to-noise ratio is terrible.
</div>

In [ ]:
# Run the agent loop — the $600 refund will trigger escalation
services = make_services()
callbacks = build_callbacks()
print("Running agent loop for C003 $600 refund (will escalate)...")
result = run_agent_loop(
    client,
    services,
    user_message,
    get_system_prompt(),
    callbacks=callbacks,
)
print(f"Stop reason: {result.stop_reason}")
print(f"Tool calls: {[tc['name'] for tc in result.tool_calls]}")

In [ ]:
# Anti-pattern: dump the full messages list as raw JSON
raw_output = format_raw_handoff(result.messages)
raw_len = len(raw_output)

# Show truncated output to illustrate the noise
PREVIEW = 500
preview = raw_output[:PREVIEW]
remaining = raw_len - PREVIEW

print(f"RAW HANDOFF OUTPUT ({raw_len:,} chars total):")
print(preview)
if remaining > 0:
    print(f"... {remaining:,} more chars")
print()
print(f"NOTE: Human agent receives {raw_len:,} chars of raw JSON")
print("      Includes: tool_use blocks, tool IDs, tool_result payloads, API artifacts")

<div style="border-left: 4px solid #dc3545; padding: 12px 16px; background: #fff5f5; margin: 8px 0;">
<strong>ANTI-PATTERN RESULT:</strong> Human agent receives thousands of chars of raw JSON including API-internal <code>tool_use</code> blocks, tool IDs, and <code>tool_result</code> payloads. The actual escalation reason and customer context are buried in the noise.
</div>

## Correct Pattern: Structured EscalationRecord via tool_choice

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Why this works:</strong> The agent loop uses <code>tool_choice={"type": "tool", "name": "escalate_to_human"}</code> to force Claude to call the escalation tool with a structured payload. The tool stores a clean <code>EscalationRecord</code> in the <code>escalation_queue</code> — 8 key fields, no noise.
</div>

**`tool_choice` enforcement** is CCA's key mechanism:
1. Callback detects blocked refund → sets `action_required = "escalate_to_human"` in result
2. Agent loop detects this signal (`_has_escalation_required`)
3. Second API call with `tool_choice` forces structured escalation
4. `EscalationRecord` is stored in `escalation_queue` — the human handoff payload

In [ ]:
# Reuse result from above — same run triggered both anti-pattern and correct pattern
# The escalation_queue already has the structured record
all_escalations = services.escalation_queue.get_all_escalations()
print(f"Escalation queue entries: {len(all_escalations)}")

if all_escalations:
    escalation = all_escalations[-1]
    escalation_dict = escalation.model_dump()
    structured_output = json.dumps(escalation_dict, indent=2, default=str)
    structured_len = len(structured_output)

    print(f"\nSTRUCTURED HANDOFF (EscalationRecord — {structured_len:,} chars):")
    print(structured_output)
else:
    structured_len = 0
    print("No escalation record found — check that $600 amount triggers the > $500 rule")

In [ ]:
class _UsageWrapper:
    def __init__(self, u):
        self.usage = u


print_usage(_UsageWrapper(result.usage))

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>CORRECT PATTERN RESULT:</strong> Human agent receives a clean structured <code>EscalationRecord</code> — 8 key fields: customer_id, customer_tier, issue_type, escalation_reason, refund_amount, priority, timestamp, agent_notes. Zero API noise.
</div>

## Compare: Raw Dump vs Structured Handoff

In [ ]:
if all_escalations and structured_len > 0:
    ratio = raw_len / structured_len
    print("Handoff Payload Comparison")
    print("=" * 45)
    print(f"{'Metric':<30} {'Anti-Pattern':>12} {'Correct':>12}")
    print("-" * 45)
    print(f"{'Payload size (chars)':<30} {raw_len:>12,} {structured_len:>12,}")
    print(f"{'Size ratio':<30} {'1.0x':>12} {f'1/{ratio:.1f}x':>12}")
    print(f"{'Contains tool_use blocks':<30} {'YES':>12} {'NO':>12}")
    print(f"{'Contains tool IDs':<30} {'YES':>12} {'NO':>12}")
    print(f"{'Key fields at top':<30} {'BURIED':>12} {'8 fields':>12}")
    print("-" * 45)
    print(f"\nRaw dump is {ratio:.1f}x larger than structured record.")
    print(f"Raw dump contains 'tool_use': {'tool_use' in raw_output}")
    print(f"Structured record contains 'tool_use': {'tool_use' in structured_output}")
else:
    print("Comparison skipped — no escalation record found.")
    compare_results(
        {"raw_len": raw_len, "has_tool_use_noise": "tool_use" in raw_output},
        {"raw_len": 0, "has_tool_use_noise": False},
    )

> **CCA Exam Tip:** Structured JSON handoffs, not raw conversation dumps. Use `tool_choice` enforcement for deterministic structured output.
>
> - Raw conversation dumps include API-internal `tool_use` blocks, tool IDs, and `tool_result` payloads — noise that human agents cannot use
> - `tool_choice={"type": "tool", "name": "escalate_to_human"}` forces Claude to produce a structured payload
> - The `EscalationRecord` has 8 clean fields: customer ID, tier, issue type, reason, amount, priority, timestamp, notes
> - **Exam signal → Answer:** "Human agents can't find the key info" → Raw handoff anti-pattern → Use structured EscalationRecord with tool_choice enforcement

## Extension: Custom PostToolUse Callback (TODO)

In [ ]:
# TODO: Write a callback that flags interactions with sentiment keywords
# HINTS:
#   1. Add sentiment keywords: ["frustrated", "terrible", "unacceptable"]
#   2. Check context["user_message"] for keywords
#   3. Set context["sentiment_flag"] = True if found
#   4. Return CallbackResult(action="allow") — flag only, don't block
# EXPECTED: Angry customer message sets sentiment_flag in context
def student_sentiment_callback(*args, **kwargs):
    from customer_service.agent.callbacks import CallbackResult

    return CallbackResult(action="allow")  # placeholder — always allows


# Guard: notebook uses default callbacks if TODO not implemented
try:
    # Student replaces the function body above, then uncomment:
    # custom_callbacks = build_callbacks()
    # custom_callbacks["log_interaction"] = student_sentiment_callback
    raise NotImplementedError("TODO: implement sentiment callback")
except NotImplementedError:
    print("TODO not yet implemented - using default callbacks. Core scenario still runs above.")